In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# Create a project folder on Drive
DRIVE_DIR = "/content/drive/MyDrive/LLM_Projects/Medical_AI_QLoRA"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Drive directory created/ready at: {DRIVE_DIR}")

Drive directory created/ready at: /content/drive/MyDrive/LLM_Projects/Medical_AI_QLoRA


In [4]:
!pip install --no-deps unsloth "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets huggingface_hub

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-3i9qhtxs/unsloth_712af31c814d47f39858bc6231a415fb
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-3i9qhtxs/unsloth_712af31c814d47f39858bc6231a415fb
  Resolved https://github.com/unslothai/unsloth.git to commit 0d9952f8ef9b3f4eaf6300706608a80f2396a77d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 5.1 MB/s eta 0:00:00
  Created wheel for unsloth: filename=unsloth-2026.9.4-py3-none-any.whl size=7728553 sha256=e45118d7997ae6022486ad5c93d9cd0ee5ec84f7c121d92d3deafb885539f19e
  Stored in directory: /tmp/pip-ephem-wheel-cache-zffwo7ak/wheels/d5/36/1d/4e65996c5b80c84a5ac1b0ba10718bdc155f8dd04352746a8f
Successfully built unsloth
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━

In [7]:
pip install unsloth_zoo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 110.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found e

In [2]:
#authenicating to the hugging face hub
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
except Exception:
    login()

# Domain Spefic Finetuning Using QLORA 4Bit

In [3]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

In [5]:
# 1. Load 4-bit Base Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.9.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [6]:
# 2. Configure PEFT QLoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                         # LoRA Rank
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],                              # Apply LoRA to all attention & MLP projection layers
    lora_alpha = 16,                # LoRA Scaling factor
    lora_dropout = 0,               # Optimized to 0 for Unsloth speedups
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Crucial for QLoRA VRAM efficiency
    random_state = 3407,
)

Unsloth 2026.9.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [7]:
# 3. Load & Format Dataset for SFT
def format_prompts(examples):
    inputs = examples["instruction"]
    outputs = examples["output"]
    texts = []
    for input_text, output_text in zip(inputs, outputs):
        text = f"<|im_start|>system\nYou are a specialized medical AI assistant.<|im_end|>\n<|im_start|>user\n{input_text}<|im_end|>\n<|im_start|>assistant\n{output_text}<|im_end|>"
        texts.append(text)
    return { "text" : texts }

sft_dataset = load_dataset("medalpaca/medical_meadow_medical_flashcards", split = "train")
sft_dataset = sft_dataset.map(format_prompts, batched = True)

README.md:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

medical_meadow_wikidoc_medical_flashcard(…): reconstructing file:   0%|          |  0.00B / 17.7MB            

medical_meadow_wikidoc_medical_flashcard(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/33955 [00:00<?, ? examples/s]

Map:   0%|          | 0/33955 [00:00<?, ? examples/s]

In [10]:
 #Configure QLoRA SFT Trainer
sft_args = SFTConfig(
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 10,
    max_steps = 120,                # Train duration (increase to 300+ for full runs)
    learning_rate = 2e-4,
    logging_steps = 10,
    output_dir = f"{DRIVE_DIR}/sft_checkpoints",
    optim = "adamw_8bit",           # 8-bit optimizer to conserve memory
    seed = 3407,
)

In [11]:
sft_trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = sft_dataset,
    args = sft_args,
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/33955 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [12]:
print("Domain-Specific QLoRA SFT ")
sft_trainer.train()

#Saving QLoRA SFT Adapters to Google Drive
sft_adapter_path = f"{DRIVE_DIR}/medical_qlora_sft_adapter"
model.save_pretrained(sft_adapter_path)
tokenizer.save_pretrained(sft_adapter_path)
print(f"training complete and sft adapters are saved to google drive")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Domain-Specific QLoRA SFT 


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 33,955 | Num Epochs = 1 | Total steps = 120
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,3.183872
20,1.856012
30,1.441449
40,1.512172
50,1.444117
60,1.442587
70,1.374306
80,1.357090
90,1.316431
100,1.422918


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/LLM_Projects/Medical_AI_QLoRA/sft_checkpoints/checkpoint-120/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/LLM_Projects/Medical_AI_QLoRA/medical_qlora_sft_adapter/tokenizer_config.json.


training complete and sft adapters are saved to google drive


# Preference Alignment (DPO) using QLoRA

In [13]:
from trl import DPOTrainer, DPOConfig

In [14]:
# Clear GPU cache before starting DPO
torch.cuda.empty_cache()

In [15]:
sft_adapter_path = f"{DRIVE_DIR}/medical_qlora_sft_adapter"
max_seq_length = 2048

In [16]:
# 1. Load SFT-adapted QLoRA model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = sft_adapter_path,   # Loads fine tuned lora adapter models in google drive
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.9.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [17]:
# Attach PEFT LoRA Adapters for Preference Alignment
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

Unsloth: Already have LoRA adapters! We shall skip this step.


In [18]:
# 3. Load and Format DPO Dataset
raw_dpo_dataset = load_dataset("argilla/distilabel-intel-orca-dpo-pairs", split = "train")

README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 79.2MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/12859 [00:00<?, ? examples/s]

In [19]:
def format_dpo_pairs(batch):
    prompts = [
        f"<|im_start|>system\nYou are a specialized medical AI assistant.<|im_end|>\n<|im_start|>user\n{p}<|im_end|>\n<|im_start|>assistant\n"
        for p in batch["input"]
    ]
    return {
        "prompt": prompts,
        "chosen": batch["chosen"],
        "rejected": batch["rejected"],
    }

dpo_dataset = raw_dpo_dataset.select(range(500)).map(format_dpo_pairs, batched = True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [21]:
# 4. Configure QLoRA DPO Trainer
dpo_args = DPOConfig(
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    warmup_steps = 0.1,
    max_steps = 60,
    learning_rate = 5e-6,           # Lower learning rate for DPO stability
    beta = 0.1,                      # DPO loss penalty parameter
    logging_steps = 5,
    optim = "adamw_8bit",
    output_dir = f"{DRIVE_DIR}/dpo_checkpoints",
    seed = 3407,
)

In [22]:
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,                # Implicit reference model managed by Unsloth
    args = dpo_args,
    train_dataset = dpo_dataset,
    tokenizer = tokenizer,
    max_length = max_seq_length,
    max_prompt_length = 512,
)

Extracting prompt in train dataset (num_proc=2):   0%|          | 0/500 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=2):   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/500 [00:00<?, ? examples/s]

In [23]:
print("QLoRA Direct Preference Optimization (DPO)")
dpo_trainer.train()

# 5. Save Final Aligned QLoRA Adapter to Drive
final_dpo_path = f"{DRIVE_DIR}/medical_qlora_dpo_final_adapter"
model.save_pretrained(final_dpo_path)
tokenizer.save_pretrained(final_dpo_path)
print(f"DPO Phase Complete! Final Aligned Adapter saved to Drive: {final_dpo_path}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


QLoRA Direct Preference Optimization (DPO)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
5,0.598086,-0.932986,-1.620145,0.675000,0.687159,-323.829895,-250.361938,-0.531264,-0.798190
10,0.675347,-1.073834,-1.637254,0.725000,0.563420,-162.905487,-202.887527,-0.717898,-0.676231
15,0.444453,-1.020004,-2.191023,0.750000,1.171019,-188.089325,-220.313354,-0.573963,-0.683249
20,0.771383,-1.157197,-1.723899,0.625000,0.566702,-283.394958,-286.960388,-0.747595,-0.884162
25,0.656827,-1.210228,-2.031682,0.675000,0.821455,-252.721725,-234.569458,-0.493988,-0.793695
30,0.502430,-0.943350,-2.121561,0.775000,1.178210,-195.501602,-251.408936,-0.611213,-0.687545
35,0.600294,-1.101634,-2.079138,0.750000,0.977503,-207.105011,-234.900024,-0.498252,-0.605416
40,0.315896,-0.711177,-2.452208,0.850000,1.741031,-114.422203,-182.327179,-0.494425,-0.522043
45,0.546975,-0.885498,-2.358914,0.725000,1.473416,-262.997742,-280.504211,-0.552151,-0.704388
50,0.405345,-0.680360,-2.285508,0.825000,1.605148,-160.205917,-203.614731,-0.599091,-0.788596


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/LLM_Projects/Medical_AI_QLoRA/dpo_checkpoints/checkpoint-60/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/LLM_Projects/Medical_AI_QLoRA/medical_qlora_dpo_final_adapter/tokenizer_config.json.


DPO Phase Complete! Final Aligned Adapter saved to Drive: /content/drive/MyDrive/LLM_Projects/Medical_AI_QLoRA/medical_qlora_dpo_final_adapter


# Export & Upload to Hugging Face Hub

In [24]:
HF_USERNAME = "ahmadnawaz21"
REPO_NAME = "Qwen2.5-1.5B-Medical-QLoRA-DPO"
FULL_REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"

In [25]:
# Merge QLoRA weights into base model & upload FP16 weights
print("Merging QLoRA weights and pushing 16-bit model to HF Hub...")
model.push_to_hub_merged(
    FULL_REPO_ID,
    tokenizer,
    save_method = "merged_16bit",
    token = userdata.get('HF_TOKEN')
)
print(f"Uploaded successfully! View your model here: https://huggingface.co/{FULL_REPO_ID}")

Merging QLoRA weights and pushing 16-bit model to HF Hub...


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in ahmadnawaz21/Qwen2.5-1.5B-Medical-QLoRA-DPO/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...oRA-DPO/model.safetensors:   1%|          | 15.9MB / 3.09GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:59<00:00, 119.25s/it]


Unsloth: Merge process complete. Saved to `/content/ahmadnawaz21/Qwen2.5-1.5B-Medical-QLoRA-DPO`
Uploaded successfully! View your model here: https://huggingface.co/ahmadnawaz21/Qwen2.5-1.5B-Medical-QLoRA-DPO


# Checking Inference

In [29]:
from unsloth.chat_templates import get_chat_template

FastLanguageModel.for_inference(model) # Enables optimized fast inference mode

messages = [
    {"role": "system", "content": "You are a specialized medical AI assistant."},
    {"role": "user", "content": "What are the key clinical features and initial management steps for acute appendicitis?"}
]

# Set the chat template for the tokenizer
tokenizer = get_chat_template(tokenizer)

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 256,
    use_cache = True,
    temperature = 0.3,
    top_p = 0.9
)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("--- Model Inference Output ---")
print(response)

Unsloth: Restored added_tokens_decoder metadata in /content/_unsloth_sentencepiece_temp/tokenizer_lme2eow6/tokenizer_config.json.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Model Inference Output ---
The key clinical features of acute appendicitis include abdominal pain that starts in the lower right quadrant and worsens with deep breathing or coughing, nausea and vomiting, and a tender mass in the lower right quadrant of the abdomen. Initial management steps for acute appendicitis include identifying the cause of the pain, such as a urinary tract infection or diverticulitis, and initiating treatment with antibiotics if necessary. If the diagnosis is confirmed, surgery is the standard treatment for acute appendicitis. Patients with acute appendicitis may also require pain management and supportive care, such as intravenous fluids and pain medication. It is important to seek medical attention promptly if abdominal pain is severe or accompanied by other symptoms, such as fever or nausea.
